# 客製化 API：Custom Layer、Loss、Metric、Callback

Keras 的高階 API 沒有現成的元件時，可以自己寫：

| 段 | 內容 | 自己寫的方式 |
|---|---|---|
| 1 | 環境與資料 | |
| 2 | Custom Layer | 繼承 `Layer`，覆寫 `build`、`call` |
| 3 | Custom Loss | 寫成函式 |
| 4 | Custom Metric | 繼承 `Metric`，覆寫 `update_state`、`result`、`reset_state` |
| 5 | Custom Callback | 繼承 `Callback`，覆寫 `on_epoch_end` |
| 6 | 實驗 | 同一個網路寫兩次：Keras 高階 API 與客製化 API，比較結果 |

每一段都有「核對」格：用簡報上手算過的數字驗證自己寫的元件是對的，不對會直接報錯。

**執行方式**：由上往下逐格執行。實驗那一段想快一點，先換 GPU（「執行階段 → 變更執行階段類型 → T4 GPU」）。
第 1 段那一格要在 `import tensorflow` 之前執行；已經執行過別的格，請「執行階段 → 重新啟動工作階段」再從頭來。

## 1. 環境與資料

In [ ]:
import os
import sys
import urllib.request

# 課本是 TF 2.0 時代寫的，Colab 內建新版 Keras 3；用舊版 Keras（tf_keras）才與課本程式一致
if 'tensorflow' in sys.modules and os.environ.get('TF_USE_LEGACY_KERAS') != '1':
    raise RuntimeError('這一格之前已經載入 TensorFlow：請「執行階段 → 重新啟動工作階段」，再從這一格開始')
os.environ['TF_USE_LEGACY_KERAS'] = '1'
try:
    import tf_keras  # noqa: F401
except ImportError:
    os.system('pip install -q tf_keras')

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow', tf.__version__)
print('GPU', tf.config.list_physical_devices('GPU'))

DATA = 'cifar10.npz'
SUBSET = 'cifar10_subset.npz'
URL = ('https://raw.githubusercontent.com/pychang-ai/115-1_DLMA/main/'
       'topics/custom-api/cifar10_subset.npz')
if os.path.exists(DATA):
    path = DATA
    print('使用已上傳的', DATA)
else:
    if not os.path.exists(SUBSET):
        print('左側沒有', DATA, '，從教材的 GitHub repo 下載縮小版')
        urllib.request.urlretrieve(URL, SUBSET)
    path = SUBSET
d = np.load(path)
# 90% 訓練、10% 驗證，與課本 train[:90%]、train[90%:] 相同
n = len(d['x_train']) * 9 // 10
x_train, y_train = d['x_train'][:n], d['y_train'][:n]
x_valid, y_valid = d['x_train'][n:], d['y_train'][n:]
x_test, y_test = d['x_test'], d['y_test']
print('訓練', x_train.shape, '驗證', x_valid.shape, '測試', x_test.shape)


def make_ds(x, y, shuffle=False, augment=False, batch_size=64):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(len(x))

    def fn(img, label):
        img = tf.cast(img, tf.float32) / 255.
        if augment:
            img = tf.image.random_flip_left_right(img)
        return img, tf.one_hot(label, 10)

    return ds.map(fn, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_data = make_ds(x_train, y_train, shuffle=True, augment=True)
valid_data = make_ds(x_valid, y_valid)
test_data = make_ds(x_test, y_test)

## 2. Custom Layer

繼承 `tf.keras.layers.Layer`。`build` 在第一次看到輸入形狀時建立權重，`call` 寫前向計算。
這裡自己寫一個卷積層：`tf.nn.conv2d` 做卷積，加偏差，再過 ReLU。

In [ ]:
class CustomConv2D(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size, strides=(1, 1), padding="VALID", **kwargs):
        super(CustomConv2D, self).__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.strides = (1, *strides, 1)
        self.padding = padding

    def build(self, input_shape):
        kernel_h, kernel_w = self.kernel_size
        input_dim = input_shape[-1]
        self.w = self.add_weight(name='kernel',
                                 shape=(kernel_h, kernel_w, input_dim, self.filters),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='bias', shape=(self.filters,),
                                 initializer='zeros', trainable=True)

    def call(self, inputs):
        x = tf.nn.conv2d(inputs, self.w, self.strides, padding=self.padding)
        x = tf.nn.bias_add(x, self.b)
        return tf.nn.relu(x)

### 核對：輸出形狀與參數量，並與內建 `Conv2D` 比較

簡報第 8、9 頁：32×32×3 的輸入接 `CustomConv2D(64, (3, 3))`，輸出 30×30×64，參數量 3×3×3×64 + 64 = 1,792。
再把同一組權重放進內建 `Conv2D`，兩者對同一張圖的輸出應該一模一樣。

In [ ]:
x = tf.random.uniform((2, 32, 32, 3), seed=0)
mine = CustomConv2D(64, (3, 3))
y_mine = mine(x)
assert y_mine.shape == (2, 30, 30, 64), y_mine.shape
n_params = sum(int(np.prod(w.shape)) for w in mine.weights)
assert n_params == 1792, n_params

ref = layers.Conv2D(64, 3, activation='relu')
ref(x)
ref.set_weights(mine.get_weights())
diff = float(tf.reduce_max(tf.abs(ref(x) - y_mine)))
assert diff < 1e-5, diff
print('輸出形狀', y_mine.shape, '參數量', n_params, '與內建 Conv2D 最大差距', diff)

## 3. Custom Loss

損失函數就是一個函式：輸入答案與預測，回傳每個樣本的損失。
`from_logits` 的寫法是把網路最後一層（沒有 softmax）的輸出直接交給 `softmax_cross_entropy_with_logits`。

In [ ]:
def custom_categorical_crossentropy(y_true, y_pred):
    return tf.nn.softmax_cross_entropy_with_logits(labels=y_true, logits=y_pred)

### 核對：簡報第 10 頁的手算例

logit = (2.0, 1.0, 0.1)，答案是第 1 類。softmax 後第 1 類機率 0.6590，損失 = −ln 0.6590 = 0.4170。

In [ ]:
logit = tf.constant([[2.0, 1.0, 0.1]])
onehot = tf.constant([[1.0, 0.0, 0.0]])
loss = float(custom_categorical_crossentropy(onehot, logit)[0])
builtin = float(keras.losses.CategoricalCrossentropy(from_logits=True)(onehot, logit))
assert abs(loss - 0.4170) < 5e-5, loss
assert abs(loss - builtin) < 1e-6, (loss, builtin)
print('自訂損失', round(loss, 4), '內建 CategoricalCrossentropy(from_logits=True)', round(builtin, 4))

## 4. Custom Metric

指標要跨批次累計，所以用兩個「變數」記：答對幾張、總共幾張。
`update_state` 每批累加，`result` 回傳答對 ÷ 總數，每個週期開始時 `reset_state` 歸零。

課本原本寫 `reset_states`（多一個 s）；新版 Keras 已改名 `reset_state`，這裡用新名稱，
舊名稱會跳出「請改名」的警告。

In [ ]:
class CustomCategoricalAccuracy(tf.keras.metrics.Metric):
    def __init__(self, name='custom_categorical_accuracy', **kwargs):
        super(CustomCategoricalAccuracy, self).__init__(name=name, **kwargs)
        self.correct = self.add_weight(name='correct_numbers', initializer='zeros')
        self.total = self.add_weight(name='total_numbers', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.argmax(y_true, axis=-1)
        y_pred = tf.argmax(y_pred, axis=-1)
        values = tf.cast(tf.equal(y_true, y_pred), tf.float32)
        self.correct.assign_add(tf.reduce_sum(values))
        self.total.assign_add(tf.cast(tf.size(values), tf.float32))

    def result(self):
        return tf.math.divide_no_nan(self.correct, self.total)

    def reset_state(self):
        self.correct.assign(0.)
        self.total.assign(0.)

### 核對：簡報第 12、13 頁的兩個批次

批次 1 有 4 張答對 3 張，result = 3/4 = 0.75；批次 2 答對 2 張，累計 5/8 = 0.625；歸零後又從頭算。

In [ ]:
CLASSES = ['貨輪', '漁船', '遊艇']
batches = [
    ([0, 1, 2, 1], [[2.0, 0.5, 0.1], [0.2, 1.8, 0.4], [1.5, 0.3, 0.9], [0.1, 2.2, 0.6]]),
    ([2, 0, 1, 2], [[0.3, 0.2, 1.7], [0.4, 1.1, 0.2], [0.6, 0.9, 0.3], [1.2, 0.1, 0.8]]),
]
m = CustomCategoricalAccuracy()
results = []
for ans, logits in batches:
    m.update_state(tf.one_hot(ans, 3), tf.constant(logits))
    results.append(float(m.result()))
assert results == [0.75, 0.625], results
m.reset_state()
assert float(m.result()) == 0.0
print('逐批累計的 result', results, '歸零後', float(m.result()))

## 5. Custom Callback

回呼函數在訓練的固定時機被呼叫。這裡的 `SaveModel` 在每個週期結束時看監控的指標，
只有比目前最好的還好才存檔。`monitor` 的名稱必須和指標的 `name` 一字不差，驗證集的值前面加 `val_`。

In [ ]:
class SaveModel(tf.keras.callbacks.Callback):
    def __init__(self, weights_file, monitor='loss', mode='min', save_weights_only=False):
        super(SaveModel, self).__init__()
        self.weights_file = weights_file
        self.monitor = monitor
        self.mode = mode
        self.save_weights_only = save_weights_only
        self.best = np.inf if mode == 'min' else -np.inf

    def save_model(self):
        if self.save_weights_only:
            self.model.save_weights(self.weights_file)
        else:
            self.model.save(self.weights_file)

    def on_epoch_end(self, epoch, logs=None):
        monitor_value = logs.get(self.monitor)
        if self.mode == 'min' and monitor_value < self.best:
            self.save_model()
            self.best = monitor_value
        elif self.mode == 'max' and monitor_value > self.best:
            self.save_model()
            self.best = monitor_value

### 核對：簡報第 14 頁

5 個週期的驗證準確率是 0.52、0.61、0.58、0.66、0.65，應該在第 1、2、4 個週期存檔，共 3 次。

In [ ]:
saved = []
cb = SaveModel('unused.h5', monitor='val_acc', mode='max')
cb.save_model = lambda: saved.append(cb.best)       # 不真的存檔，只記下被呼叫的時機
epochs_saved = []
for k, v in enumerate([0.52, 0.61, 0.58, 0.66, 0.65], 1):
    before = len(saved)
    cb.on_epoch_end(k - 1, {'val_acc': v})
    if len(saved) > before:
        epochs_saved.append(k)
assert epochs_saved == [1, 2, 4], epochs_saved
print('存檔的週期', epochs_saved, '共', len(epochs_saved), '次')

## 6. 實驗：同一個網路寫兩次

model_1 用 Keras 高階 API；model_2 把四個元件都換成自己寫的。兩者的結構與參數量應該完全一樣（941,066）。

與課本的兩處不同，都是為了讓比較公平：

1. model_1 的 `ModelCheckpoint` 加上 `save_best_only=True`，和 `SaveModel` 一樣只存最好的一個週期；課本沒設，會每個週期都存，最後留下的是最後一個週期。
2. `monitor` 名稱拼成 `val_categorical_accuracy`；課本寫成 `val_catrgorical_accuracy`（少了一個 o），Keras 找不到這個名稱，`mode='max'` 就沒作用。

週期數預設 10。課本是 100，GPU 上要跑一小時以上；CPU 太慢，請換 GPU。

In [ ]:
EPOCHS = 10
os.makedirs('lab6-logs', exist_ok=True)


def conv_net(conv, name):
    inputs = keras.Input(shape=(32, 32, 3))
    x = conv(64)(inputs)
    x = layers.MaxPool2D()(x)
    for f in (128, 256, 128, 64):
        x = conv(f)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10)(x)
    return keras.Model(inputs, outputs, name=name)


model_1 = conv_net(lambda f: layers.Conv2D(f, 3, activation='relu', kernel_initializer='glorot_uniform'), 'model-1')
model_2 = conv_net(lambda f: CustomConv2D(f, (3, 3)), 'model-2')
assert model_1.count_params() == model_2.count_params() == 941066
print('兩個模型的參數量', model_1.count_params())

### model_1：Keras 高階 API

In [ ]:
model_1.compile(keras.optimizers.Adam(),
                loss=keras.losses.CategoricalCrossentropy(from_logits=True),
                metrics=[keras.metrics.CategoricalAccuracy()])
h1 = model_1.fit(train_data, epochs=EPOCHS, validation_data=valid_data,
                 callbacks=[keras.callbacks.ModelCheckpoint('lab6-logs/save.h5',
                                                            monitor='val_categorical_accuracy',
                                                            mode='max', save_best_only=True,
                                                            save_weights_only=True)])

### model_2：客製化 API

In [ ]:
model_2.compile(keras.optimizers.Adam(),
                loss=custom_categorical_crossentropy,
                metrics=[CustomCategoricalAccuracy()])
h2 = model_2.fit(train_data, epochs=EPOCHS, validation_data=valid_data,
                 callbacks=[SaveModel('lab6-logs/custom_save.h5',
                                      monitor='val_custom_categorical_accuracy',
                                      mode='max', save_weights_only=True)])

## 7. 結果整理

載入兩個模型各自存下的最佳權重，在測試集上評估。兩者結構相同、只差在元件的來源，
準確率應該很接近；不會一模一樣，因為初始權重與資料洗牌都有隨機性。

In [ ]:
import pandas as pd

model_1.load_weights('lab6-logs/save.h5')
model_2.load_weights('lab6-logs/custom_save.h5')
loss_1, acc_1 = model_1.evaluate(test_data, verbose=0)
loss_2, acc_2 = model_2.evaluate(test_data, verbose=0)
pd.DataFrame({'模型': ['model_1 高階 API', 'model_2 客製化 API'],
              '最佳驗證準確率': [max(h1.history['val_categorical_accuracy']),
                           max(h2.history['val_custom_categorical_accuracy'])],
              '測試損失': [loss_1, loss_2], '測試準確率': [acc_1, acc_2]}).round(4)

## 練習

1. 簡報第 13 頁的批次 2 若再多答對 1 張，`result` 是多少？（答案 6/8 = 0.75，可以改上面的核對格驗證）
2. `CustomConv2D(32, (5, 5))` 接在 32×32×3 的輸入後，輸出形狀與參數量各是多少？（答案 28×28×32，2,432）
3. 作業 W08：自訂一個 Metric（計算正確分類樣本數），加進 `model_2.compile` 的 `metrics`，截圖每個週期的輸出。